<a href="https://colab.research.google.com/github/sarshadad-codeee/FlyRank_ML_Task1/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/sarshadad-codeee/FlyRank_ML_Task1"
REPO_DIR = "FlyRank_ML_Task1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working directory:", os.getcwd())
!pip install duckdb --quiet

Working directory: /content/FlyRank_ML_Task1


In [2]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

base = "hf://datasets/FlyRank/internship-warehouse"
march_path = f"{base}/fact_content_daily_performance/month=2026-03/*.parquet"
april_path = f"{base}/fact_content_daily_performance/month=2026-04/*.parquet"

In [3]:
march_df = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
        SUM(gsc_clicks) AS march_clicks,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_sum_position) AS march_sum_position
    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()
march_df["avg_position"] = march_df["march_sum_position"] / march_df["march_impressions"]
march_df["ctr"] = march_df["march_clicks"] / march_df["march_impressions"].replace(0, pd.NA)

april_df = con.sql(f"""
    SELECT content_hash_id, client_hash_id, SUM(gsc_clicks) AS april_clicks
    FROM read_parquet('{april_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

merged = march_df.merge(april_df, on=["content_hash_id", "client_hash_id"], how="inner")
merged["is_declining"] = (merged["april_clicks"] < merged["march_clicks"]).astype(int)
print(f"Shape: {merged.shape}, base rate: {merged['is_declining'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape: (158549, 9), base rate: 0.279


In [4]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

feature_cols = ["march_clicks", "march_impressions", "avg_position", "ctr"]
X = merged[feature_cols].fillna(0)
y = merged["is_declining"]
groups = merged["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X.iloc[train_idx], y.iloc[train_idx])

merged["decline_risk_score"] = rf.predict_proba(X)[:, 1]
print("Model retrained on honest split, scores attached to full dataset.")

Model retrained on honest split, scores attached to full dataset.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [5]:
# Build archetypes from the model's own risk score + real signal buckets
merged["reason_code"] = "monitor"
merged.loc[(merged["decline_risk_score"] >= 0.6) & (merged["ctr"] < merged["ctr"].median()), "reason_code"] = "high_risk_low_ctr"
merged.loc[(merged["decline_risk_score"] >= 0.6) & (merged["avg_position"] >= 20), "reason_code"] = "high_risk_weak_position"
merged.loc[(merged["decline_risk_score"] >= 0.4) & (merged["decline_risk_score"] < 0.6), "reason_code"] = "moderate_risk_watch"

action_map = {
    "high_risk_low_ctr": "review_for_ctr_fix",
    "high_risk_weak_position": "review_for_position_and_ctr",
    "moderate_risk_watch": "monitor_next_cycle",
    "monitor": "no_action_needed",
}
merged["action"] = merged["reason_code"].map(action_map)

ranked_queue = merged.sort_values("decline_risk_score", ascending=False).reset_index(drop=True)

print(f"Total scored: {len(ranked_queue)}")
print(ranked_queue["reason_code"].value_counts())
print()
ranked_queue[["content_hash_id", "client_hash_id", "decline_risk_score", "reason_code", "action"]].head(10)

Total scored: 158549
reason_code
monitor                    141971
moderate_risk_watch          9015
high_risk_weak_position      7563
Name: count, dtype: int64



,content_hash_id,client_hash_id,decline_risk_score,reason_code,action
0,content_6eec3d6a7af49b23,client_3ffa76342f366962,0.946870,monitor,no_action_needed
1,content_7a987d7bc5fddeeb,client_f623b01661d4bfe4,0.946338,monitor,no_action_needed
2,content_8cc2b3be12f3864b,client_3ffa76342f366962,0.946254,monitor,no_action_needed
3,content_507653893440072c,client_f623b01661d4bfe4,0.946251,high_risk_weak_position,review_for_position_and_ctr
4,content_1cfa05a210b14c50,client_f623b01661d4bfe4,0.945917,monitor,no_action_needed
5,content_3fe9ea218b00792e,client_3ffa76342f366962,0.945651,monitor,no_action_needed
6,content_3617e20335510220,client_f623b01661d4bfe4,0.945061,monitor,no_action_needed
7,content_bdeb90d74d3a8597,client_3ffa76342f366962,0.944132,monitor,no_action_needed
8,content_f5eaab83140ac94e,client_3ffa76342f366962,0.943752,monitor,no_action_needed
9,content_d3f982a535c1f35f,client_3ffa76342f366962,0.943669,monitor,no_action_needed


In [7]:
"""
Ranked actions + reason codes:

The queue is ranked by decline_risk_score (the validated model's
predicted probability of a click decline, from ML-08/ML-09 --
Precision@50 = 0.780 on an honest, client-grouped split).

Reason codes translate the score into words a reviewer can act on,
combining risk level with the specific signal driving it:
- high_risk_weak_position: high decline risk + weak avg_position (>=20)
  -> action: review_for_position_and_ctr
- high_risk_low_ctr: high decline risk + below-median CTR -> action:
  review_for_ctr_fix
- moderate_risk_watch: moderate risk, no dominant single signal ->
  action: monitor_next_cycle
- monitor: low risk -> action: no_action_needed

Archetype -> action mapping is deliberately coarse (4 categories, not
a unique reason per page) so a human reviewer can learn the pattern
quickly rather than treating every row as a one-off judgment call.

Decay/refresh insight (framed per the writing-honest-claims skill's
claim ladder): in this dataset, pages flagged high_risk_weak_position
show an OBSERVED association between weak average position and lower
CTR (established in ML-07's signal check: a ~10x CTR drop from
position 1-3 to position 50+). This is DECISION-SUPPORT for
prioritizing review, not a causal claim that fixing position will
increase CTR by any specific amount -- no controlled experiment was
run.
Observed note: 0 rows landed in high_risk_low_ctr on its own -- every
high-risk row this run also had a weak avg_position, so the later
assignment (high_risk_weak_position) captured all of them. This is
consistent with ML-07's signal audit finding that position and CTR are
strongly linked in this data, not a coding error. If a future dataset
slice produces high-risk, well-positioned, low-CTR pages, this reason
code would activate then.
"""

"\nRanked actions + reason codes:\n\nThe queue is ranked by decline_risk_score (the validated model's \npredicted probability of a click decline, from ML-08/ML-09 -- \nPrecision@50 = 0.780 on an honest, client-grouped split).\n\nReason codes translate the score into words a reviewer can act on, \ncombining risk level with the specific signal driving it:\n- high_risk_weak_position: high decline risk + weak avg_position (>=20) \n  -> action: review_for_position_and_ctr\n- high_risk_low_ctr: high decline risk + below-median CTR -> action: \n  review_for_ctr_fix\n- moderate_risk_watch: moderate risk, no dominant single signal -> \n  action: monitor_next_cycle\n- monitor: low risk -> action: no_action_needed\n\nArchetype -> action mapping is deliberately coarse (4 categories, not \na unique reason per page) so a human reviewer can learn the pattern \nquickly rather than treating every row as a one-off judgment call.\n\nDecay/refresh insight (framed per the writing-honest-claims skill's \ncl

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [8]:
"""
Intended use: this playbook is DECISION-SUPPORT for a content reviewer
deciding which pages to look at first each cycle, for Lane 2
(refresh/content opportunity scoring). It is not a fully-automated
action system -- every flagged page still requires a human to read the
"why" and decide the specific fix.

Who should use it: a content strategist or SEO reviewer with FlyRank
context on the client, using this queue to triage a limited weekly
review budget (e.g. "which 50 pages do I look at this week").

Where it stops being valid:
1. Time window: built on March->April 2026 data for one warehouse
   snapshot. Not validated on other months, other years, or other
   markets/seasons -- re-validation needed before reuse on a different
   period.
2. Client coverage: only ~55 of 104 total clients had usable data this
   month (per ML-04's data-limit finding). This queue is silent, not
   necessarily accurate, for clients with sparse or missing history.
3. Feature scope: based on 4 observable signals only (clicks,
   impressions, position, CTR). Does not account for real-world causes
   like a site migration, a manual penalty, seasonal demand, or a
   competitor's content change -- a human reviewer supplies that
   context, the model cannot.
4. Precision, not perfection: Precision@50 = 0.780 on the validated
   split means roughly 1 in 5 flagged pages in the top 50 will NOT
   actually be declining -- an expected, honest error rate, not a flaw
   to hide.
"""

'\nIntended use: this playbook is DECISION-SUPPORT for a content reviewer \ndeciding which pages to look at first each cycle, for Lane 2 \n(refresh/content opportunity scoring). It is not a fully-automated \naction system -- every flagged page still requires a human to read the \n"why" and decide the specific fix.\n\nWho should use it: a content strategist or SEO reviewer with FlyRank \ncontext on the client, using this queue to triage a limited weekly \nreview budget (e.g. "which 50 pages do I look at this week").\n\nWhere it stops being valid:\n1. Time window: built on March->April 2026 data for one warehouse \n   snapshot. Not validated on other months, other years, or other \n   markets/seasons -- re-validation needed before reuse on a different \n   period.\n2. Client coverage: only ~55 of 104 total clients had usable data this \n   month (per ML-04\'s data-limit finding). This queue is silent, not \n   necessarily accurate, for clients with sparse or missing history.\n3. Feature 

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.